# WaSPS-DTW — Interactive Classification Notebook

This notebook lets you:
1. **Generate or load** a dataset (synthetic exponential / CPAZMaL HDF5)
2. **Estimate** distribution parameters at each timestep
3. **Classify** with 4 methods (eucl_params, eucl_raw, wasps, sta) × 2 modes (KNN + barycenter)
4. **Visualise** barycenters, confusion matrices, and sensitivity curves

Run from the **repo root** with the `.venv` kernel activated.

In [1]:
import sys
from pathlib import Path
import jax

# Add src/ to path — no pip install needed
_SRC = Path("src").resolve()
sys.path.insert(0, str(_SRC))

import sys
from pathlib import Path

_SRC = Path("../src").resolve()
sys.path.insert(0, str(_SRC))

# Enable JAX x64 globally — must come BEFORE any jax.numpy import
jax.config.update("jax_enable_x64", True)

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, f1_score, ConfusionMatrixDisplay

import distributions
from data.preprocess import clean_time_series, to_fixed_n
from costs import SqEuclidean, WaSPS
from softdtw import SoftDTW
from classification.barycenter_clf import fit_barycenters, predict
from classification.nn import knn_predict as sdtw_knn
from baselines.sta_wrapper import knn_predict as sta_knn

print(f"JAX devices: {jax.devices()}")
print("Imports OK")

JAX devices: [CpuDevice(id=0)]
Imports OK


---
## 1. Dataset

Choose `DATASET_TYPE`: `"synthetic"` or `"cpazmal"`.  
For `"cpazmal"`, set `HDF5_PATH` to your local file.

In [2]:
# ── Configuration knobs ──────────────────────────────────────────────────────
DATASET_TYPE    = "cpazmal"   # "synthetic" | "cpazmal"
FAMILY          = "weibull" # "exponential" | "weibull" (cpazmal → weibull)
SEED            = 42

# Synthetic only
N_CLASSES            = 3
N_TRAIN_PER_CLASS    = 8
N_TEST_PER_CLASS     = 4
T                    = 6     # number of timesteps
N_SAMPLES            = 50    # samples per timestep
RATES                = [1.0, 3.0, 8.0]   # rate β per class (length == N_CLASSES)

# CPAZMaL only
HDF5_PATH   = "/home/mgallet/Documents/Codes/Python/1_DONE/CPAZMAL/DATASET/dataset_original/PAZTSX_CRYO_ML.hdf5"
MAX_GROUPS_PER_CLASS = None   # None = all groups; integer = per-class cap for a quick smoke-test
# ─────────────────────────────────────────────────────────────────────────────

In [3]:
rng = np.random.default_rng(SEED)

if DATASET_TYPE == "synthetic":
    assert FAMILY == "exponential", "synthetic only supports exponential"
    assert len(RATES) == N_CLASSES

    train_raw, test_raw, train_labels, test_labels = [], [], [], []
    for cls, rate in enumerate(RATES):
        for _ in range(N_TRAIN_PER_CLASS):
            train_raw.append(rng.exponential(1.0 / rate, (T, N_SAMPLES)))
            train_labels.append(cls)
        for _ in range(N_TEST_PER_CLASS):
            test_raw.append(rng.exponential(1.0 / rate, (T, N_SAMPLES)))
            test_labels.append(cls)
    train_labels = np.array(train_labels)
    test_labels  = np.array(test_labels)
    class_names  = [f"class_{i} (β={r})" for i, r in enumerate(RATES)]
    T_data       = T

elif DATASET_TYPE == "cpazmal":
    FAMILY = "weibull"
    from sklearn.model_selection import train_test_split
    from data.cpazmal_loader import MLDatasetLoader, extract_time_series
    loader = MLDatasetLoader(HDF5_PATH)
    # New K-fold-classification contract (HAG excluded by default via exclude_classes):
    # one continuous full-year series per window, no separate train/predict period.
    data    = extract_time_series(loader, max_groups_per_class=MAX_GROUPS_PER_CLASS, window_size=8)
    X       = list(data["X"])
    labels  = np.asarray(data["y"])
    groups  = np.asarray(data["groups"])
    cn_dict = data.get("class_names", {})  # int → class-name string
    class_names = [cn_dict.get(c, str(c)) for c in sorted(set(labels.tolist()))]

    # Single stratified holdout split (mirrors data_utils.py::_load_cpazmal's n_splits=1
    # path) — not group-aware; use configs/cpazmal.yaml's K-fold runner for a rigorous split.
    idx_train, idx_test = train_test_split(
        np.arange(len(labels)), test_size=0.2, random_state=SEED, stratify=labels,
    )
    train_raw    = [X[i] for i in idx_train]
    test_raw     = [X[i] for i in idx_test]
    train_labels = labels[idx_train]
    test_labels  = labels[idx_test]
    T_data       = train_raw[0].shape[0]

else:
    raise ValueError(f"Unknown DATASET_TYPE: {DATASET_TYPE}")

print(f"Train: {len(train_raw)} series, Test: {len(test_raw)} series")
print(f"Classes: {class_names}")
print(f"T={T_data}, sample shape example: {train_raw[0].shape}")


CPAZMaL: Time Series Extraction (for WaSPS-DTW classification)
  Window:      8×8
  Orbit:       DSC  |  Polarisation: HH
  Period:      20200101 – 20201231
  Scale type:  amplitude
  Excluded:    ['STUDY', 'HAG']
  Mask max:    ≤1 (10.0%)



Groups:   0%|          | 0/61 [00:00<?, ?grp/s, ABL001 (ABL)]/home/mgallet/Documents/Codes/Python/3_DEVELOPPEMENT/WaSPS-DTW/WaSPS-DTW/src/data/cpazmal_loader.py:352: RuntimeWarning: invalid value encountered in sqrt
  images = np.where(images >= 0, np.sqrt(images), np.nan).astype(np.float32)
Groups: 100%|██████████| 61/61 [01:00<00:00,  1.00grp/s, ROC009 (ROC)]


Extraction complete:
  Total samples (windows):  18225
  X[0].shape:               (29, 64)  (T, W²)
    Class  0 (ACC            ): 5090 samples
    Class  1 (PLA            ): 1354 samples
    Class  2 (ROC            ): 2218 samples
    Class  3 (ABL            ): 3785 samples
    Class  4 (ICA            ): 249 samples
    Class  5 (LAC            ): 163 samples
    Class  6 (FOR            ): 5366 samples
Train: 14580 series, Test: 3645 series
Classes: ['ACC', 'PLA', 'ROC', 'ABL', 'ICA', 'LAC', 'FOR']
T=29, sample shape example: (29, 64)


In [4]:
np.unique(train_labels, return_counts=True), np.unique(test_labels, return_counts=True), len(train_raw), len(test_raw)

((array([0, 1, 2, 3, 4, 5, 6], dtype=int32),
  array([4072, 1083, 1775, 3028,  199,  130, 4293])),
 (array([0, 1, 2, 3, 4, 5, 6], dtype=int32),
  array([1018,  271,  443,  757,   50,   33, 1073])),
 14580,
 3645)

---
## 1b. CPAZMaL data exploration (run only when DATASET_TYPE == "cpazmal")

This section investigates hypotheses about why CPAZMaL classification performance is limited:

1. **Geographic group imbalance** — with `max_train_samples` stratified by class only, a class's
   training samples may all come from a single geographic group → non-representative barycenter.
2. **NaN / invalid pixels** — SAR windows with nodata/negative values reduce effective N per timestep.
3. **Parametric fit quality** — does Weibull (MLE vs log-cumulant) actually fit the SAR pixel
   distribution? A high KS statistic indicates poor fit.
4. **Barycenter stability** — with very few training samples (5–20), are barycenters stable?

Cells below use the `data` dict from Cell 4 (which includes `groups` and `group_names`).
Skip this section for synthetic data.

In [5]:
# ── 1b-0: Guard — only run for CPAZMaL ──────────────────────────────────────
if DATASET_TYPE != "cpazmal":
    print("DATASET_TYPE is not 'cpazmal' — skipping exploration section.")
else:
    # The `data` dict from Cell 4 already contains the full extraction result.
    # Re-use it here (no re-extraction needed).
    from data.cpazmal_loader import MLDatasetLoader
    _loader  = MLDatasetLoader(HDF5_PATH)
    _groups  = np.asarray(data["groups"])           # (N,) int — geographic group per sample
    _gnames  = data.get("group_names", {})           # {int → str}
    _cnames  = data.get("class_names", {})           # {int → str}

    # ── Dataset summary ──────────────────────────────────────────────────────
    _classes = sorted(set(labels.tolist()))
    print(f"=== CPAZMaL dataset overview ===")
    print(f"Total samples: {len(labels)}  |  Classes: {len(_classes)}")
    for cls in _classes:
        mask = labels == cls
        n    = mask.sum()
        cls_groups = _groups[mask]
        n_geo = len(set(cls_groups.tolist()))
        cname = _cnames.get(cls, str(cls))
        print(f"  class {cls:2d} ({cname:20s}): {n:3d} samples  |  {n_geo} geographic groups")

    print(f"\nSeries shape (train example): {train_raw[0].shape}  (T, W²)")
    print(f"T_train={T_data}, W²={train_raw[0].shape[1]} (window {int(train_raw[0].shape[1]**0.5)}×{int(train_raw[0].shape[1]**0.5)})")

=== CPAZMaL dataset overview ===
Total samples: 18225  |  Classes: 7
  class  0 (ACC                 ): 5090 samples  |  9 geographic groups
  class  1 (PLA                 ): 1354 samples  |  7 geographic groups
  class  2 (ROC                 ): 2218 samples  |  9 geographic groups
  class  3 (ABL                 ): 3785 samples  |  7 geographic groups
  class  4 (ICA                 ): 249 samples  |  8 geographic groups
  class  5 (LAC                 ): 163 samples  |  9 geographic groups
  class  6 (FOR                 ): 5366 samples  |  9 geographic groups

Series shape (train example): (29, 64)  (T, W²)
T_train=29, W²=64 (window 8×8)


In [ ]:
# ── 1b-3: Fit quality — histogram + PDF (MLE vs log-cumulant) (Hypothesis 3) 
# Uses plot_samples_with_fitted_pdf from src/plot
if DATASET_TYPE == "cpazmal":
    from plot.classification_plots import plot_samples_with_fitted_pdf
    import scipy.stats as scipy_stats
    from data.preprocess import clean_series as _cs

    # Show 2 classes × 3 timesteps inline (inline display rather than saving PDF)
    _cls_show = _classes[:min(3, len(_classes))]
    print(f"=== Weibull fit quality: MLE vs log-cumulant for classes {_cls_show} ===")
    print("(Generating one figure per class — 3 timesteps per figure)\n")

    for cls in _cls_show:
        # train_labels (not labels): aligned with train_raw after the train/test split
        # in cell b543be26 — indexing with the full-dataset `labels` here caused an
        # IndexError (index out of bounds for the smaller train_raw list).
        mask   = train_labels == cls
        series = [train_raw[i] for i in np.where(mask)[0]]
        cname  = _cnames.get(cls, str(cls))
        # Save to a temp dir (and show inline via matplotlib)
        import tempfile, os
        with tempfile.TemporaryDirectory() as _tmpdir:
            plot_samples_with_fitted_pdf(
                series, 'weibull', cls, cname,
                output_dir='tmp/', n_timesteps=3, save_pdf=True,
            )
        # Rerun inline (save_pdf=False, output_dir=None → plt.show needed)
        plot_samples_with_fitted_pdf(
            series, 'weibull', cls, cname,
            output_dir=None, n_timesteps=3, save_pdf=False,
        )
plt.show()


In [ ]:
# ── 1b-4: Barycenter stability vs N training samples (Hypothesis 4) ─────────
# Refit the wasps barycenter for one class at N=5/10/20/all training samples.
# Large divergence between curves → unstable barycenter (too few samples).
if DATASET_TYPE == "cpazmal":
    from classification.barycenter_clf import fit_barycenters as _fb

    _GAMMA_EXPLORE = 1.0   # local defaults (GAMMA/LR defined later in cell-14)
    _LR_EXPLORE    = 0.1
    _N_STEPS_EXPLORE = 50  # reduced for quick exploration

    _cls_demo  = _classes[0]   # change this index to inspect a different class
    cname_demo = _cnames.get(_cls_demo, str(_cls_demo))
    print(f"=== Barycenter stability (class {_cls_demo}: {cname_demo}) ===")
    print(f"Fitting Weibull params → wasps barycenter at N=5/10/20/all (γ={_GAMMA_EXPLORE}, lr={_LR_EXPLORE}) …\n")

    # Estimate params for all training samples of this class
    # train_labels (not labels): aligned with train_raw — see the same fix in cell ff214e58.
    mask_demo  = train_labels == _cls_demo
    all_series = [train_raw[i] for i in np.where(mask_demo)[0]]
    dist_w2    = distributions.get('weibull')
    all_params = [dist_w2.fit_time_series(clean_time_series(s), dtype=np.float64)
                  for s in all_series]
    n_all = len(all_params)
    print(f"Total samples for this class: {n_all}")

    wasps_sdtw_bary_expl = SoftDTW(
        WaSPS('weibull', use_positivity_constraint=True),
        _GAMMA_EXPLORE, is_divergence=True, manual_grad=True,
    )

    n_caps    = sorted(set([5, 10, 20, n_all]))
    bary_by_n = {}
    for nc in n_caps:
        subset  = all_params[:nc]
        sub_lbl = np.zeros(nc, dtype=int)
        b = _fb(subset, sub_lbl, wasps_sdtw_bary_expl,
                n_steps=_N_STEPS_EXPLORE, lr=_LR_EXPLORE, verbose=True, n_jobs=-1)
        bary_by_n[nc] = b[0]   # class 0 in the per-class fit
        print(f"  N={nc:3d}: shape={bary_by_n[nc].shape}  "
              f"k_mean={bary_by_n[nc][:,0].mean():.3f}  λ_mean={bary_by_n[nc][:,1].mean():.3f}")

    # Plot: k (shape) and λ (scale) over time for each N
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3), sharex=True)
    colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(n_caps)))
    for (nc, bary), col in zip(bary_by_n.items(), colors):
        ax1.plot(bary[:, 0], color=col, lw=1.5, label=f'N={nc}')
        ax2.plot(bary[:, 1], color=col, lw=1.5, label=f'N={nc}')
    ax1.set_title(f'k (shape) — class {_cls_demo}', fontsize=9)
    ax2.set_title(f'λ (scale) — class {_cls_demo}', fontsize=9)
    for ax in (ax1, ax2):
        ax.set_xlabel('Timestep')
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)
    plt.suptitle(f'Barycenter stability vs N training samples — {cname_demo}',
                 fontsize=9, fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## 2. Parameter estimation

Converts each raw `(T, N)` series → `(T, p)` parameter array:  
- Exponential: `p=1`, column = rate β  
- Weibull: `p=2`, columns = (k, λ)

In [6]:
print(f"Estimating {FAMILY} parameters …")
dist = distributions.get(FAMILY)
train_params = [dist.fit_time_series(clean_time_series(s), dtype=np.float64) for s in train_raw]
test_params  = [dist.fit_time_series(clean_time_series(s), dtype=np.float64) for s in test_raw]

print(f"Parameter shape: {train_params[0].shape}  (T, p)")

# Quick sanity: plot mean param over time for each class
classes = sorted(set(train_labels.tolist()))
fig, ax = plt.subplots(figsize=(8, 3))
for cls in classes:
    cls_params = np.stack([p for p, l in zip(train_params, train_labels) if l == cls])
    mean_p = cls_params.mean(axis=0)  # (T, n_params)
    ax.plot(mean_p[:, 0], label=class_names[cls] if cls < len(class_names) else str(cls))
ax.set_xlabel("Timestep")
ax.set_ylabel("Mean param[:,0]")
ax.set_title("Mean estimated parameter (col 0) per class")
ax.legend()
plt.tight_layout()
plt.show()

Estimating weibull parameters …


KeyboardInterrupt: 

---
## 3. Method configuration

In [ ]:
# ── Classification hyperparameters ───────────────────────────────────────────
GAMMA       = 1.0     # Soft-DTW smoothing (cpazmal.yaml)
N_STEPS     = 200     # barycenter optimisation steps (cpazmal.yaml)
LR          = 0.005   # learning rate (cpazmal.yaml; default optimizer is sgd, not adam)
STA_EPSILON = 0.05    # Sinkhorn regularisation for sta
KNN_K       = 1       # k for KNN
# ─────────────────────────────────────────────────────────────────────────────

<cell_type>markdown</cell_type>---
## 4. Classification: 4 methods × 2 modes (KNN + barycenter)

| Method | Representation | Cost | Notes |
|--------|----------------|------|-------|
| `eucl_params` | MLE params `(T, p)` | SqEuclidean | no positivity constraint |
| `eucl_raw` | raw samples `(T, N)` | SqEuclidean | no sign constraint |
| `wasps` | MLE params `(T, p)` | WaSPS W₂² | positivity constraint + manual grad |
| `sta` | raw samples `(T, N)` | OT Sinkhorn | slow; KNN only here |

In [ ]:
import time


import jax.numpy as jnp
def predict2(
    test_series: list,
    barycenters: dict,
    cost_fn,
    gamma: float,
) -> np.ndarray:
    """Classify each test series by minimum SoftDTW divergence to class barycenters.

    Args:
        test_series:  List of (T, p) arrays.
        barycenters:  dict label → (T, p) array (from fit_barycenters).
        cost_fn:      Ground cost callable(a, b) → scalar (positive-param space).
        gamma:        SoftDTW regularisation.

    Returns:
        predictions: (N_test,) array of class labels.
    """
    classes = sorted(barycenters.keys())
    bary_jax = {cls: jnp.array(barycenters[cls]) for cls in classes}
    print(classes)
    # Plain SDTW (not divergence): avoids self-term ½SDTW(b,b) dominating when
    # T_test ≠ T_bary, and matches KNN semantics (nearest centroid by DTW distance).
    sdtw_fn = SoftDTW(cost_fn, gamma, is_divergence=True, manual_grad=True)

    @jax.jit
    def divergence_to_bary(z: jax.Array, b: jax.Array) -> jax.Array:
        return sdtw_fn.value(z, b)

    preds = []
    for s in test_series:
        z = jnp.array(s)
        dists = [float(divergence_to_bary(z, bary_jax[cls])) for cls in classes]
        preds.append(classes[int(np.argmin(dists))])
    return np.array(preds)

results = {}

# Two-instance pattern for WaSPS:
#   barycenter: use_positivity_constraint=True  (θ-space optimisation, log_correction auto-set)
#   KNN/predict: log_correction=True only       (params already positive, divergence ≥ 0)
wasps_cost_bary = WaSPS(FAMILY, use_positivity_constraint=True)
wasps_cost_knn  = WaSPS(FAMILY, log_correction=True)

# SoftDTW instances for barycenter fitting (auto-sets log_correction on WaSPS)
sdtw_eucl  = SoftDTW(SqEuclidean(), GAMMA, is_divergence=True, manual_grad=False)
sdtw_wasps = SoftDTW(wasps_cost_bary, GAMMA, is_divergence=True, manual_grad=True)

# ── eucl_params: SqEuclidean on estimated params ─────────────────────────────
print("[eucl_params] SqEuclidean on params (barycenter + KNN) …")
t0 = time.time()
bary_eucl = fit_barycenters(train_params, train_labels, sdtw_eucl, n_steps=N_STEPS, lr=LR, verbose=True, n_jobs=-1)
preds_eucl_bary = predict2(test_params, bary_eucl, SqEuclidean(), GAMMA)
results["eucl_params/bary"] = {"preds": preds_eucl_bary, "bary": bary_eucl, "time": time.time() - t0}
# preds_eucl_knn  = sdtw_knn(train_params, train_labels, test_params,
#                              cost_fn=SqEuclidean(), gamma=GAMMA, k=KNN_K)
# results["eucl_params/knn"] = {"preds": preds_eucl_knn, "time": 0}
print(f"  bary F1={f1_score(test_labels, preds_eucl_bary, average='weighted', zero_division=0):.3f}  ")
      # f"knn F1={f1_score(test_labels, preds_eucl_knn, average='weighted', zero_division=0):.3f}  ({time.time()-t0:.1f}s)")

# # ── eucl_raw: SqEuclidean on raw samples ─────────────────────────────────────
print("[eucl_raw] SqEuclidean on raw samples (barycenter + KNN) …")
t0 = time.time()
bary_raw = fit_barycenters(train_raw, train_labels, sdtw_eucl, n_steps=N_STEPS, lr=LR)
preds_raw_bary = predict(test_raw, bary_raw, SqEuclidean(), GAMMA)
results["eucl_raw/bary"] = {"preds": preds_raw_bary, "bary": bary_raw, "time": time.time() - t0}
# preds_raw_knn  = sdtw_knn(train_raw, train_labels, test_raw,
                        #    cost_fn=SqEuclidean(), gamma=GAMMA, k=KNN_K)
# results["eucl_raw/knn"] = {"preds": preds_raw_knn, "time": 0}
print(f"  bary F1={f1_score(test_labels, preds_raw_bary, average='weighted', zero_division=0):.3f}  ")
    #   f"knn F1={f1_score(test_labels, preds_raw_knn, average='weighted', zero_division=0):.3f}  ({time.time()-t0:.1f}s)")
# 
# ── wasps: WaSPS W₂² on params ────────────────────────────────────────────────
print("[wasps] WaSPS W₂² on params (barycenter + KNN) …")
t0 = time.time()
bary_wasps = fit_barycenters(train_params, train_labels, sdtw_wasps, n_steps=N_STEPS, lr=LR, verbose=True, n_jobs=-1)
preds_wasps_bary = predict2(test_params, bary_wasps, wasps_cost_knn, GAMMA)
results["wasps/bary"] = {"preds": preds_wasps_bary, "bary": bary_wasps, "time": time.time() - t0}
# preds_wasps_knn  = sdtw_knn(train_params, train_labels, test_params,
#                              cost_fn=wasps_cost_knn, gamma=GAMMA, k=KNN_K)
# results["wasps/knn"] = {"preds": preds_wasps_knn, "time": 0}
print(f"  bary F1={f1_score(test_labels, preds_wasps_bary, average='weighted', zero_division=0):.3f}  ")
      # f"knn F1={f1_score(test_labels, preds_wasps_knn, average='weighted', zero_division=0):.3f}  ({time.time()-t0:.1f}s)")

# # ── sta: OT Sinkhorn KNN (barycenter excluded here — too slow for interactive use) ──
# if FAMILY != "weibull":
#     print("[sta] STA KNN (Sinkhorn OT + SoftDTW) — may take several minutes …")
#     t0 = time.time()
#     preds_sta = sta_knn(train_raw, train_labels, test_raw,
#                         gamma=GAMMA, epsilon=STA_EPSILON, k=KNN_K)
#     results["sta/knn"] = {"preds": preds_sta, "time": time.time() - t0}
#     print(f"  knn F1={f1_score(test_labels, preds_sta, average='weighted', zero_division=0):.3f}  ({time.time()-t0:.1f}s)")
# else:
#     print("[sta] skipped (Weibull)")

---
## 5. Results summary

In [ ]:
results['eucl_params/bary'].keys(),name

In [ ]:
# Confusion matrices
n_methods = len(results)
fig, axes = plt.subplots(1, n_methods, figsize=(4 * n_methods, 4))
if n_methods == 1:
    axes = [axes]

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(test_labels, res["preds"])
    disp = ConfusionMatrixDisplay(cm)
    disp.plot(ax=ax, colorbar=False)
    pred = res["preds"] 
    f1 = f1_score(test_labels, pred, average='weighted', zero_division=0)
    ax.set_title(f"{name} - F1 = {f1:.3f}", fontsize=8)

plt.suptitle("Confusion matrices", y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Barycenter visualisation

In [ ]:
# Plot wasps barycenters vs. class mean
fig, axes = plt.subplots(1, len(classes), figsize=(4 * len(classes), 3), sharey=True)
if len(classes) == 1:
    axes = [axes]

param = 0
for ax, cls in zip(axes, classes):
    cls_params = np.stack([p for p, l in zip(train_params, train_labels) if l == cls])
    for p in cls_params:
        ax.plot(p[:, param], color="steelblue", alpha=0.3, linewidth=0.8)
    ax.plot(cls_params.mean(0)[:, param], color="steelblue", linewidth=1.5, linestyle="--",
            label="class mean")
    ax.plot(bary_wasps[cls][:, param], color="crimson", linewidth=2, label="wasps barycenter")
    label = class_names[cls] if cls < len(class_names) else str(cls)
    ax.set_title(label, fontsize=9)
    ax.set_xlabel("Timestep")
    ax.legend(fontsize=7)

axes[0].set_ylabel("Param col 0")
plt.suptitle("WaSPS barycenters (crimson) vs individual series (blue)")
plt.tight_layout()
plt.show()

---
## 7. Sensitivity to γ

In [ ]:
# Sweep γ for eucl_params and wasps (fast — no STA involved)
GAMMAS = [0.1, 0.5, 1.0, 2.0, 5.0]

f1_eucl_g, f1_wasps_g = [], []
for g in GAMMAS:
    sdtw_e = SoftDTW(SqEuclidean(), g, is_divergence=True, manual_grad=False)
    sdtw_w = SoftDTW(WaSPS(FAMILY, use_positivity_constraint=True), g,
                     is_divergence=True, manual_grad=True)

    b_e = fit_barycenters(train_params, train_labels, sdtw_e, n_steps=N_STEPS, lr=LR)
    p_e = predict(test_params, b_e, SqEuclidean(), g)
    f1_eucl_g.append(f1_score(test_labels, p_e, average="weighted", zero_division=0))

    b_w = fit_barycenters(train_params, train_labels, sdtw_w, n_steps=N_STEPS, lr=LR)
    p_w = predict(test_params, b_w, WaSPS(FAMILY, log_correction=True), g)
    f1_wasps_g.append(f1_score(test_labels, p_w, average="weighted", zero_division=0))
    print(f"  γ={g:.2f}  eucl_params={f1_eucl_g[-1]:.3f}  wasps={f1_wasps_g[-1]:.3f}")

plt.figure(figsize=(6, 3))
plt.plot(GAMMAS, f1_eucl_g, "o-", label="eucl_params")
plt.plot(GAMMAS, f1_wasps_g, "s-", label="wasps")
plt.xlabel("γ (Soft-DTW regularisation)")
plt.ylabel("F1 weighted")
plt.xscale("log")
plt.legend()
plt.title("Sensitivity to γ")
plt.tight_layout()
plt.show()

In [ ]:
import json

# Matches DATASET_TYPE="cpazmal" above + configs/cpazmal.yaml's output.dir;
# path is relative to analysis/ (this notebook's cwd) — hence the ../ prefix.
result_path = Path("../results/jax_cpazmal/classification_full.json")
if result_path.exists():
    with open(result_path) as f:
        saved = json.load(f)
    print(f"Config: {saved['config']['dataset']}")
    print(f"\n{'Method':12s}  {'Mode':10s}  {'F1 mean':>8}  {'F1 std':>7}  {'Acc mean':>9}")
    print("-" * 55)
    for r in saved["summary"]:
        print(f"  {r['method']:12s}/{r['mode']:10s}  "
              f"f1={r['f1_mean']:.3f}±{r['f1_std']:.3f}  "
              f"acc={r['acc_mean']:.3f}±{r['acc_std']:.3f}")
else:
    print(f"{result_path} not found. Run experiments/run_classification.py first.")

In [ ]:
import pandas as pd

sens_dir   = Path("results/jax_sensitivity")
gamma_csv  = sens_dir / "sensitivity_gamma.csv"
ntrain_csv = sens_dir / "sensitivity_ntrain.csv"

if gamma_csv.exists() and ntrain_csv.exists():
    df_g = pd.read_csv(gamma_csv)
    df_n = pd.read_csv(ntrain_csv)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))

    for m in [c for c in df_g.columns if c != "gamma"]:
        ax1.plot(df_g["gamma"], df_g[m], "o-", label=m)
    ax1.set_xlabel("γ")
    ax1.set_ylabel("F1 weighted (KNN, k=1)")
    ax1.set_xscale("log")
    ax1.set_title("Sensitivity to γ")
    ax1.legend(fontsize=8)

    for m in [c for c in df_n.columns if c != "n_train"]:
        ax2.plot(df_n["n_train"], df_n[m], "o-", label=m)
    ax2.set_xlabel("n_train (per class)")
    ax2.set_ylabel("F1 weighted (KNN, k=1)")
    ax2.set_title("Sensitivity to n_train")
    ax2.legend(fontsize=8)

    plt.suptitle("Sensitivity analysis — synthetic exponential", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("Run  python experiments/run_sensitivity.py --output-dir results/jax_sensitivity  first.")

In [ ]:
# Generate one custom series and classify it
custom_rate = 2.0   # change this to test different rates
custom_raw  = rng.exponential(1.0 / custom_rate, (T_data, N_SAMPLES))
dist = distributions.get(FAMILY)
custom_params = dist.fit_time_series(clean_time_series(custom_raw), dtype=np.float64)

# Classify with eucl_params and wasps
pred_eucl  = predict([custom_params], bary_eucl,  SqEuclidean(), GAMMA)[0]
pred_wasps = predict([custom_params], bary_wasps, WaSPS(FAMILY, log_correction=True), GAMMA)[0]

print(f"Custom series (rate={custom_rate}):")
print(f"  eucl_params → class {pred_eucl}  ({class_names[pred_eucl] if pred_eucl < len(class_names) else pred_eucl})")
print(f"  wasps       → class {pred_wasps} ({class_names[pred_wasps] if pred_wasps < len(class_names) else pred_wasps})")